# Distribution des choix de mobilité — un passage Mistral

Objectif : soumettre **100 prompts** au modèle Mistral avec une **température unique** (1.0),  
puis afficher la répartition des modes de transport choisis.

In [ ]:
import sys
import json
import time
import random
import yaml
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

# ── Chemins ────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../..')
sys.path.insert(0, str(PROJECT_ROOT))

EXCHANGES_FILE = PROJECT_ROOT / 'experiments' / 'current' / 'llm_exchanges.jsonl'
SCHEMAS_FILE   = PROJECT_ROOT / 'llm_module' / 'prompts' / 'schemas.json'
PROMPTS_FILE   = Path('prompts.yaml')

SYSTEM_PROMPT_PREFIX = 'Tu es un expert en mobilité urbaine à Toulouse, France.'
CATEGORY             = 'itinary_multi_agent'
SAMPLE_SIZE          = 100

# ── Persona système ─────────────────────────────────────────────────────────
PERSONA_KEY = 'persona_v5'
with open(PROMPTS_FILE) as f:
    _prompts_cfg = yaml.safe_load(f)
PERSONA_SYSTEM_PROMPT = _prompts_cfg['prompts'][PERSONA_KEY]['content']
print(f'Persona chargé : {PERSONA_KEY!r} ({len(PERSONA_SYSTEM_PROMPT)} caractères)')

# ── Configuration ──────────────────────────────────────────────────────────
PROVIDER_NAME = 'mistral'
TEMPERATURE   = 1.0

OUTPUT_CSV = Path('results') / f'single_run_{PROVIDER_NAME}_T{TEMPERATURE}.csv'
print(f'Provider     : {PROVIDER_NAME}')
print(f'Température  : {TEMPERATURE}')
print(f'Output CSV   : {OUTPUT_CSV}')

## 1 — Chargement du JSONL et échantillonnage

In [ ]:
def parse_multiline_jsonl(path: Path) -> list:
    content = path.read_text(encoding='utf-8').strip()
    decoder = json.JSONDecoder()
    entries, pos = [], 0
    while pos < len(content):
        stripped = content[pos:].lstrip()
        pos += len(content[pos:]) - len(stripped)
        if not stripped:
            break
        obj, offset = decoder.raw_decode(stripped)
        entries.append(obj)
        pos += offset
    return entries


all_entries = parse_multiline_jsonl(EXCHANGES_FILE)

eligible = [
    e for e in all_entries
    if e.get('messages', [{}])[0].get('content', '').startswith(SYSTEM_PROMPT_PREFIX)
]

if len(eligible) > SAMPLE_SIZE:
    random.seed(42)
    eligible = random.sample(eligible, SAMPLE_SIZE)

print(f'Entrées totales   : {len(all_entries)}')
print(f'Entrées éligibles : {len(eligible)} (max {SAMPLE_SIZE})')

## 2 — Chargement du schéma et informations du provider

In [ ]:
from llm_module.adapters.base import get_adapter
from llm_module.settings.models import InternalMessage, InternalRequest
from llm_module.tasks.llm_config import settings

with open(SCHEMAS_FILE) as f:
    RESPONSE_SCHEMA = json.load(f)[CATEGORY]

provider_cfg = settings.providers[PROVIDER_NAME]
RPM_LIMIT    = provider_cfg.rpm_limit

print(f'Provider    : {PROVIDER_NAME}')
print(f'Modèle      : {provider_cfg.default_model}')
print(f'RPM limit   : {RPM_LIMIT}')
print(f'Appels total: {len(eligible)}')
print(f'Durée estim.: ~{len(eligible) / RPM_LIMIT:.1f} min')

## 3 — Fonction d'appel avec gestion des erreurs

In [ ]:
from llm_module.adapters.base import ProviderClientError, ProviderServerError, ProviderParseError
from llm_module.worker.task_worker import _parse_ratelimit_reset_seconds

MAX_RETRIES    = 5
MAX_RETRY_WAIT = 300.0


def call_once(entry: dict) -> list:
    raw_messages = entry['messages']
    messages = [
        InternalMessage(role=raw_messages[0]['role'], content=PERSONA_SYSTEM_PROMPT),
        *[InternalMessage(role=m['role'], content=m['content']) for m in raw_messages[1:]],
    ]
    request = InternalRequest(
        provider=PROVIDER_NAME,
        messages=messages,
        response_schema=RESPONSE_SCHEMA,
        temperature=TEMPERATURE,
    )
    llm_output, _, _ = get_adapter(PROVIDER_NAME).call(request)
    user_prompt = entry['messages'][1]['content']
    return [
        {
            'user_prompt':  user_prompt,
            'agent_id':     agent.agent_id,
            'chosen_index': agent.chosen_index,
            'mode':         agent.mode,
            'reason':       agent.reason,
        }
        for agent in llm_output.agents
    ]


def call_with_retry(entry: dict) -> list:
    parse_attempts = 0
    for attempt in range(MAX_RETRIES + 1):
        try:
            return call_once(entry)
        except ProviderClientError as exc:
            if exc.status_code == 429 and attempt < MAX_RETRIES:
                wait = _parse_ratelimit_reset_seconds(exc.ratelimit_reset)
                if wait > MAX_RETRY_WAIT:
                    raise
                time.sleep(wait)
            else:
                raise
        except ProviderServerError:
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
            else:
                raise
        except ProviderParseError:
            if parse_attempts < 1:
                parse_attempts += 1
                time.sleep(2.0)
            else:
                raise

## 4 — Exécution séquentielle (rate-limité)

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

if OUTPUT_CSV.exists() and OUTPUT_CSV.stat().st_size > 100:
    existing_df  = pd.read_csv(OUTPUT_CSV, engine='python')
    done_prompts = set(existing_df['user_prompt'])
    print(f'Mode reprise : {len(done_prompts)} prompts déjà traités')
else:
    done_prompts = set()
    if OUTPUT_CSV.exists():
        OUTPUT_CSV.unlink()
    print('Nouveau run')

min_interval       = 60.0 / RPM_LIMIT
last_call_ts       = 0.0
all_rows           = []
consecutive_errors = 0
MAX_CONSECUTIVE    = 10

for entry in tqdm(eligible, desc='Mistral'):
    user_prompt = entry['messages'][1]['content']
    if user_prompt in done_prompts:
        continue

    # rate-limit
    now  = time.monotonic()
    wait = min_interval - (now - last_call_ts)
    if wait > 0:
        time.sleep(wait)
    last_call_ts = time.monotonic()

    try:
        rows = call_with_retry(entry)
        consecutive_errors = 0
        all_rows.extend(rows)
        write_header = not OUTPUT_CSV.exists()
        pd.DataFrame(rows).to_csv(OUTPUT_CSV, mode='a', header=write_header, index=False)
    except Exception as exc:
        consecutive_errors += 1
        print(f'ECHEC ({consecutive_errors}/{MAX_CONSECUTIVE}): {exc}')
        if consecutive_errors >= MAX_CONSECUTIVE:
            print('Abandon après trop d\'erreurs consécutives')
            break

print(f'Terminé — {len(all_rows)} nouvelles lignes collectées')

## 5 — Chargement et nettoyage des résultats

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_raw   = pd.read_csv(OUTPUT_CSV, engine='python')
df_valid = df_raw[df_raw['chosen_index'].between(1, 5)].copy()

print(f'Lignes totales  : {len(df_raw)}')
print(f'Lignes valides  : {len(df_valid)}')
print(f'Lignes exclues  : {len(df_raw) - len(df_valid)}')
print(f'Prompts uniques : {df_valid["user_prompt"].nunique()}')
print()
df_valid.head()

In [ ]:
# ── Palette officielle modes de transport ────────────────────────────────────
MODE_COLORS = {
    'Voiture': 'red',
    'Vélo':    'purple',
    'TC':      'green',
    'Marche':  'cyan',
    'Autre':   'gray',
}

def categorize_mode(m: str) -> str:
    if not isinstance(m, str):
        return 'Autre'
    ml = m.lower()
    if any(k in ml for k in ('voiture', 'car', 'conducteur')):
        return 'Voiture'
    if any(k in ml for k in ('vélo', 'velo', 'bicycle', 'cycling', 'vélib')):
        return 'Vélo'
    if any(k in ml for k in ('bus', 'metro', 'métro', 'tram', 'transit', 'transports en commun', 'public_transport')):
        return 'TC'
    if any(k in ml for k in ('marche', 'foot', 'walk')):
        return 'Marche'
    print(f'Mode non classifié (Autre) : {m!r}')
    return 'Autre'

df_valid['mode_cat'] = df_valid['mode'].apply(categorize_mode)

mode_counts = df_valid['mode_cat'].value_counts()
mode_pct    = (mode_counts / len(df_valid) * 100).round(1)

print('Répartition des modes :')
for mode, n in mode_counts.items():
    print(f'  {mode:<10} {n:>4} réponses  ({mode_pct[mode]:.1f}%)')

## 6 — Visualisation de la répartition

In [ ]:
cats_order = [c for c in MODE_COLORS if c in mode_counts.index]
vals       = [mode_counts.get(c, 0) for c in cats_order]
pcts       = [mode_pct.get(c, 0.0) for c in cats_order]
colors     = [MODE_COLORS[c] for c in cats_order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f'Distribution des modes de transport — Mistral T={TEMPERATURE} ({len(df_valid)} réponses valides)',
    fontsize=13, fontweight='bold'
)

# ── Graphique en barres ────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(cats_order, vals, color=colors, edgecolor='white', linewidth=0.5, width=0.6)
ax.set_ylabel('Nombre de réponses')
ax.set_xlabel('Mode de transport')
ax.set_title('Fréquences absolues')
ax.spines[['top', 'right']].set_visible(False)
for bar, v, p in zip(bars, vals, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{v}\n({p:.1f}%)', ha='center', va='bottom', fontsize=9)

# ── Camembert ─────────────────────────────────────────────────────────────
ax = axes[1]
wedge_props = {'edgecolor': 'white', 'linewidth': 1.2}
wedges, texts, autotexts = ax.pie(
    vals,
    labels=cats_order,
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=wedge_props,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_color('white')
    at.set_fontweight('bold')
ax.set_title('Répartition en pourcentage')

plt.tight_layout()
plt.savefig(f'results/single_run_distribution_T{TEMPERATURE}.png', dpi=150)
plt.show()

### Répartition des itinéraires choisis (index 1–5)

In [ ]:
index_counts = df_valid['chosen_index'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    index_counts.index.astype(int),
    index_counts.values,
    color='#5C6BC0', edgecolor='white', linewidth=0.5, width=0.6
)
ax.set_xlabel('Itinéraire choisi (index)')
ax.set_ylabel('Nombre de réponses')
ax.set_title(
    f'Répartition des itinéraires choisis — Mistral T={TEMPERATURE}',
    fontsize=12, fontweight='bold'
)
ax.set_xticks(range(1, 6))
ax.spines[['top', 'right']].set_visible(False)
for bar, v in zip(bars, index_counts.values):
    pct = v / len(df_valid) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{v}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'results/single_run_index_T{TEMPERATURE}.png', dpi=150)
plt.show()

### Tableau récapitulatif

In [ ]:
summary = (
    df_valid.groupby('mode_cat')
    .agg(
        n_responses=('mode_cat', 'count'),
        n_agents=('agent_id', 'nunique'),
        avg_chosen_index=('chosen_index', 'mean'),
        avg_reason_len=('reason', lambda s: s.str.len().mean()),
    )
    .assign(pct=lambda d: (d['n_responses'] / d['n_responses'].sum() * 100).round(1))
    [['n_responses', 'pct', 'n_agents', 'avg_chosen_index', 'avg_reason_len']]
    .sort_values('n_responses', ascending=False)
)
summary.columns = ['Réponses', '% total', 'Agents uniques', 'Index moyen', 'Long. raison moy.']
summary.index.name = 'Mode'
summary